In [1]:
import pandas as pd
from scipy.stats import spearmanr

CSV_PATH = "importance_long.csv"  # liegt im selben Ordner
TOP_K = 5

df = pd.read_csv(CSV_PATH)
FWS = ["auto-sklearn", "auto-pytorch", "autogluon"]


def top5_share(g):
    s = g.sort_values("mean_abs_shap", ascending=False)["mean_abs_shap"]
    return s.head(TOP_K).sum() / s.sum()


def spearman_pairs(pivot):
    out = {}
    fws = [f for f in FWS if f in pivot.columns]
    for i in range(len(fws)):
        for j in range(i + 1, len(fws)):
            rho, _ = spearmanr(pivot[fws[i]], pivot[fws[j]])
            out[f"{fws[i]} vs {fws[j]}"] = round(rho, 2)
    return out


def pivot_features(sub):
    return sub.pivot(index="feature", columns="framework", values="mean_abs_shap").fillna(0.0)


In [2]:
# ---------- Sex Classification ----------
sex = df[df["category"] == "Classification"]

print("=== Sex Classification ===")
print("Mean absolute SHAP:", sex.groupby("framework")["mean_abs_shap"].mean().round(4).to_dict())
print("Top-5 Share:       ", sex.groupby("framework").apply(top5_share, include_groups=False).round(2).to_dict())
print("Spearman:          ", spearman_pairs(pivot_features(sex)))


=== Sex Classification ===
Mean absolute SHAP: {'auto-pytorch': 0.0024, 'auto-sklearn': 0.0032, 'autogluon': 0.0031}
Top-5 Share:        {'auto-pytorch': 0.5, 'auto-sklearn': 0.21, 'autogluon': 0.35}
Spearman:           {'auto-sklearn vs auto-pytorch': 0.34, 'auto-sklearn vs autogluon': 0.37, 'auto-pytorch vs autogluon': 0.79}


In [3]:
# ---------- Force Regression: Top-5-Share PRO TASK (Basis fuer overall & Richtung) ----------
force = df[df["category"] == "Force regression"].copy()
force["direction"] = force["task"].str.extract(r"^predict_(F_[A-Z]+_PRO)_")

per_task_share = (
    force.groupby(["framework", "task", "direction"])
         .apply(top5_share, include_groups=False)
         .reset_index(name="top5_share")
)
per_task_share.head()


,framework,task,direction,top5_share
0,auto-pytorch,predict_F_AP_PRO_20pct_F_AP_PRO_21,F_AP_PRO,0.413091
1,auto-pytorch,predict_F_AP_PRO_40pct_F_AP_PRO_41,F_AP_PRO,0.403895
2,auto-pytorch,predict_F_AP_PRO_60pct_F_AP_PRO_61,F_AP_PRO,0.397312
3,auto-pytorch,predict_F_AP_PRO_80pct_F_AP_PRO_81,F_AP_PRO,0.442147
4,auto-pytorch,predict_F_ML_PRO_20pct_F_ML_PRO_21,F_ML_PRO,0.240123


In [4]:
# ---------- Force Regression (overall = Mittel ueber alle Tasks) ----------
print("=== Force Regression (overall) ===")
print("Mean absolute SHAP:", force.groupby("framework")["mean_abs_shap"].mean().round(3).to_dict())
print("Top-5 Share:       ", per_task_share.groupby("framework")["top5_share"].mean().round(2).to_dict())


=== Force Regression (overall) ===
Mean absolute SHAP: {'auto-pytorch': 0.001, 'auto-sklearn': 0.024, 'autogluon': 0.002}
Top-5 Share:        {'auto-pytorch': 0.35, 'auto-sklearn': 0.36, 'autogluon': 0.43}


In [8]:
# ---------- Force Regression pro Richtung (Mittel ueber die 4 Tasks je Richtung) ----------
print("=== Force Regression (pro Richtung, Top-5 Share) ===")
per_task_share.groupby(["framework", "direction"])["top5_share"].mean().round(3).unstack("direction")


=== Force Regression (pro Richtung, Top-5 Share) ===


direction,F_AP_PRO,F_ML_PRO,F_V_PRO
framework,,,
auto-pytorch,0.414,0.323,0.317
auto-sklearn,0.341,0.284,0.467
autogluon,0.458,0.379,0.455


In [6]:
# ---------- Force Regression: Spearman pro Task ----------
print("=== Force Regression: Spearman pro Task ===")
rhos = []
for task, sub in force.groupby("task"):
    r = spearman_pairs(pivot_features(sub))
    rhos.append(r)
    print(task, r)

print("\nMittel:", pd.DataFrame(rhos).mean().round(2).to_dict())


=== Force Regression: Spearman pro Task ===
predict_F_AP_PRO_20pct_F_AP_PRO_21 {'auto-sklearn vs auto-pytorch': 0.47, 'auto-sklearn vs autogluon': 0.65, 'auto-pytorch vs autogluon': 0.69}
predict_F_AP_PRO_40pct_F_AP_PRO_41 {'auto-sklearn vs auto-pytorch': 0.42, 'auto-sklearn vs autogluon': 0.41, 'auto-pytorch vs autogluon': 0.52}
predict_F_AP_PRO_60pct_F_AP_PRO_61 {'auto-sklearn vs auto-pytorch': 0.2, 'auto-sklearn vs autogluon': 0.47, 'auto-pytorch vs autogluon': 0.4}
predict_F_AP_PRO_80pct_F_AP_PRO_81 {'auto-sklearn vs auto-pytorch': 0.27, 'auto-sklearn vs autogluon': 0.55, 'auto-pytorch vs autogluon': 0.49}
predict_F_ML_PRO_20pct_F_ML_PRO_21 {'auto-sklearn vs auto-pytorch': 0.3, 'auto-sklearn vs autogluon': 0.21, 'auto-pytorch vs autogluon': 0.48}
predict_F_ML_PRO_40pct_F_ML_PRO_41 {'auto-sklearn vs auto-pytorch': -0.0, 'auto-sklearn vs autogluon': 0.34, 'auto-pytorch vs autogluon': 0.45}
predict_F_ML_PRO_60pct_F_ML_PRO_61 {'auto-sklearn vs auto-pytorch': 0.28, 'auto-sklearn vs auto